# Phase 4-A｜TSM SMA 5/20 Research-only Smoke — 正式單次執行（v1）

**Notebook 版本：`P4A-SMOKE-RUN-v1`**　｜　Agent pin `9439f8c`　｜　Core pin `b16179a`

這份 notebook 會**消耗唯一一次研究額度**（`RA-IBKR_TSM-SMA520_DEV-18350b3adba8e28a7e5e`，max_batches = 1），對 development snapshot `TSM_DEV_1M_20240529_20240628_18350b3adba8e28a7e5e` 跑一次 SMA 5/20，BASE／LOW／STRESS／CAT_STRESS 四個成本情境一個批次。

Notebook 只是 thin runner，所有邏輯都在 Agent `src/` 和 Core：
`Snapshot → Core 保守 15m → SMA 5/20 → Orders → Core ExecutionEngine → Fills → Core 成本 → Metrics → Archive`

| 儲存格 | 做什麼 | 會消耗額度？ |
|---|---|---|
| 1 | 設定 | 否 |
| 2 | **Version Sync**：remote HEAD、checkout HEAD、pin、working tree、Core 身分、notebook 版本、import 來源 | 否 |
| 3 | Drive、Snapshot、授權 0/1、GitHub 與 Drive 上都沒有先前的使用紀錄 | 否 |
| 4 | Core `renew_development_receipt` + 非消耗性 probe，必須 `COMPATIBLE_READY_TO_CONSUME` | 否 |
| 5 | **正式執行**（需要你把 `I_CONFIRM_SINGLE_BATCH` 改成 True） | **是，1/1** |
| 6 | 把使用紀錄與結果摘要推回 PR #5（Colab Secret `GITHUB_TOKEN`；沒有就下載 zip） | 否 |

任何檢查失敗都會停止，不會 reset、不會刪除、不會放寬條件。**請不要在第 5 格失敗後重跑第 5 格**：額度已消耗的失敗會寫入 `FAILURE.json`，依規約停止並回報。

結果只限 RESEARCH_ONLY，Formal Market Gate 維持 **NO_GO**；不讀 Holdout，不呼叫任何券商 API。

In [ ]:
# --- 1. 設定（不要修改 pin）---
NOTEBOOK_VERSION = "P4A-SMOKE-RUN-v1"
NOTEBOOK_PATH    = "notebooks/Phase4A_TSM_SMA520_Research_Smoke_Run_v1.ipynb"
AGENT_REPO   = "https://github.com/hh4832/strategy-research-agent.git"
BRANCH       = "feature/phase4a-tsm-research-only"
AGENT_SHA    = "9439f8c6f53b9c6c7f2f49b5a543ffdbc84f2f2a"    # 已審查、CI 綠的執行程式碼（不是「最新 HEAD」）
CORE_SHA     = "b16179a7cc077c36046574769a32c367fa476cae"
CONFIG       = "configs/research_only/tsm_sma_5_20_smoke_2024_06_dev_18350b3adba8e28a7e5e.json"
CONFIG_SHA   = "b6ac297faec9e289b422086b06eccd22d65afed5c38525a9c3eb6247e15bbfe9"
AUTH_ID      = "RA-IBKR_TSM-SMA520_DEV-18350b3adba8e28a7e5e"
SNAPSHOT_ID  = "TSM_DEV_1M_20240529_20240628_18350b3adba8e28a7e5e"
MANIFEST_SHA = "8b6fc838b72ae3015f6eb418c3a77115e53a86f185e05c2249e9dcd58192cfc5"
PARQUET_SHA  = "239d5741d65cc0c7118f618a5ebb7b1dd946f385d843cae151a24edc1e0a8452"
LINEAGE_SHA  = "3455f2b3d39f4fd05dda7ca9fed472c9ff0bcbf345f89f177e78eccbe5fd92a4"
ACTOR        = "hh4832"
SNAPSHOT     = "/content/drive/MyDrive/00Quant_Research/data/development_snapshots/" + SNAPSHOT_ID
ARCHIVE_ROOT = "/content/drive/MyDrive/Quant_Research/strategy-research-agent"
AGENT_ROOT   = "/content/strategy-research-agent-smoke"

I_CONFIRM_SINGLE_BATCH = False   # ← 第 4 格顯示 COMPATIBLE_READY_TO_CONSUME 後，改成 True，再執行第 1 格與第 5 格

import json, os, re, subprocess, sys, hashlib, importlib
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
assert sys.version_info >= (3, 11), sys.version
for s in (AGENT_SHA, CORE_SHA):
    assert re.fullmatch(r"[0-9a-f]{40}", s)
SNAPSHOT, ARCHIVE_ROOT, AGENT_ROOT = Path(SNAPSHOT), Path(ARCHIVE_ROOT), Path(AGENT_ROOT)
def git(*a, cwd=None, check=True):
    r = subprocess.run(["git", *a], cwd=cwd, text=True, capture_output=True)
    if check and r.returncode:
        raise RuntimeError("git " + a[0] + " failed: " + r.stderr[-500:])
    return r.stdout.strip()
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""): h.update(c)
    return h.hexdigest()
print("notebook", NOTEBOOK_VERSION, "| agent pin", AGENT_SHA[:7], "| core pin", CORE_SHA[:7])

In [ ]:
# --- 2. Version Sync（不消耗額度）---
# 2a. 若這個 runtime 先前已載入其他位置的 strategy_agent，舊模組會被沿用 → 停止，請 Runtime → Restart。
stale = [m for m in ("strategy_agent", "market_research_core") if m in sys.modules
         and not str(getattr(sys.modules[m], "__file__", "")).startswith((str(AGENT_ROOT), "/usr", "/opt", sys.prefix))]
assert not stale, f"STOP: 此 runtime 已載入其他來源的 {stale}；請 Runtime → Restart session 後重新 Run all"

# 2b. Agent checkout：不存在就 clone 並 detach 到 pin；存在就必須完全吻合，否則停止（不 reset、不刪除）。
if not AGENT_ROOT.exists():
    git("clone", "--quiet", AGENT_REPO, str(AGENT_ROOT))
    git("checkout", "--quiet", "--detach", AGENT_SHA, cwd=AGENT_ROOT)
    print("Fresh Agent checkout created")
else:
    print("Existing Agent checkout detected: verifying identity before reuse")
remote = git("remote", "get-url", "origin", cwd=AGENT_ROOT)
assert remote.rstrip("/").removesuffix(".git") == AGENT_REPO.removesuffix(".git"), "STOP: wrong remote " + remote
head = git("rev-parse", "HEAD", cwd=AGENT_ROOT)
status = git("status", "--porcelain", "--untracked-files=all", cwd=AGENT_ROOT)
git("fetch", "--quiet", "origin", BRANCH, cwd=AGENT_ROOT)
remote_head = git("rev-parse", "FETCH_HEAD", cwd=AGENT_ROOT)
pin_on_branch = subprocess.run(["git", "merge-base", "--is-ancestor", AGENT_SHA, remote_head], cwd=AGENT_ROOT).returncode == 0

# 2c. 快取的舊 notebook：比較 GitHub 上同名 notebook 的版本字串。
remote_nb = subprocess.run(["git", "show", f"{remote_head}:{NOTEBOOK_PATH}"], cwd=AGENT_ROOT, text=True, capture_output=True)
remote_nb_version = (re.search(r'NOTEBOOK_VERSION = \\"([^\\"]+)\\"', remote_nb.stdout) or [None, None])[1] if remote_nb.returncode == 0 else None

sync = {"notebook_version": NOTEBOOK_VERSION, "notebook_version_on_github_head": remote_nb_version,
        "github_remote_head": remote_head, "checkout_head": head, "pinned_agent_sha": AGENT_SHA,
        "pin_is_ancestor_of_remote_head": pin_on_branch, "working_tree_clean": status == "",
        "working_tree_status": status.splitlines()[:20]}
print(json.dumps(sync, indent=2))
assert head == AGENT_SHA, "STOP: checkout HEAD ≠ pin（不自行 checkout/reset；請換新的 AGENT_ROOT 或重啟 runtime）"
assert status == "", "STOP: checkout 有未提交或未追蹤的變更（若是上次執行留下的使用紀錄，代表額度已消耗，請勿重跑）"
assert pin_on_branch, "STOP: pin 不在 GitHub 分支歷史上"
assert remote_nb_version in (None, NOTEBOOK_VERSION), (
    f"STOP: 你開啟的是舊 notebook（{NOTEBOOK_VERSION}），GitHub 上是 {remote_nb_version}。請從 GitHub 重新開啟最新檔名。")
if remote_head != AGENT_SHA:
    changed = git("diff", "--name-only", AGENT_SHA, remote_head, "--", "governance", "registry", "src", "configs", cwd=AGENT_ROOT)
    print("NOTE: GitHub HEAD 比 pin 新；本次仍以 pin 執行（不自動採用最新 HEAD）。pin 之後變更的治理/程式檔：",
          changed.splitlines() or "none")
    assert not [p for p in changed.splitlines() if p.startswith(("governance/", "registry/"))], (
        "STOP: pin 之後 GitHub 上的 governance/registry 有變動（可能已有新的授權或使用紀錄）；請回報，不要執行")

# 2d. 安裝 pinned Core 與 Agent，確認 import 來源與 Core 身分。
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "market-research-core @ git+https://github.com/hh4832/market-research-core.git@" + CORE_SHA], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(AGENT_ROOT) + "[research]"], check=True)
for p in (AGENT_ROOT / "src", AGENT_ROOT):
    if str(p) not in sys.path: sys.path.insert(0, str(p))
import strategy_agent, market_research_core
from market_research_core.tsm_native import core_code_identity
ident = core_code_identity()
imports = {"strategy_agent": strategy_agent.__file__, "market_research_core": market_research_core.__file__}
print("Core identity:", ident); print("Import source:", json.dumps(imports, indent=2))
assert strategy_agent.__file__.startswith(str(AGENT_ROOT / "src")), "STOP: strategy_agent 不是從 pinned checkout 載入"
assert ident["core_git_sha"] == CORE_SHA and ident["core_tree_dirty"] is not True, "STOP: Core 身分不符"
VERSION_SYNC = sync | {"core_identity": ident, "import_source": imports}
print("VERSION SYNC: PASS")

In [ ]:
# --- 3. Drive、Snapshot、授權與先前使用紀錄（不消耗額度）---
from google.colab import drive
drive.mount("/content/drive")
os.chdir(AGENT_ROOT)
from strategy_agent.research_only.runner import ResearchOnlyRunner, canonical_sha
from strategy_agent.research_only.development_run import prior_archived_attempts
from strategy_agent.governance.research_authorization import ResearchAuthorizationBook

assert SNAPSHOT.is_dir() and SNAPSHOT.name == SNAPSHOT_ID, "STOP: development snapshot not found"
for n in ("manifest.json", "lineage.json", "tsm_development_1m.parquet", "binding.json"):
    assert (SNAPSHOT / n).is_file(), "STOP: missing " + n
assert sha256(SNAPSHOT / "manifest.json") == MANIFEST_SHA, "STOP: manifest SHA"
assert sha256(SNAPSHOT / "lineage.json") == LINEAGE_SHA, "STOP: lineage SHA"
assert json.loads((SNAPSHOT / "manifest.json").read_text())["parquet_sha256"] == PARQUET_SHA, "STOP: parquet pin"

cfg_path = AGENT_ROOT / CONFIG
cfg = json.loads(cfg_path.read_text())
assert cfg["provenance"]["config_sha256"] == canonical_sha(cfg) == CONFIG_SHA, "STOP: canonical config SHA"
assert cfg["authorization_id"] == AUTH_ID and cfg["dataset"]["snapshot_id"] == SNAPSHOT_ID
assert cfg["strategy"]["parameters"] == {"fast": 5, "slow": 20}
book = ResearchAuthorizationBook(AGENT_ROOT / "governance/research_authorizations.jsonl", AGENT_ROOT / "governance/approvers.json")
auth = book.get(AUTH_ID)
assert auth and auth["approved_by"] == "hh4832" and auth["max_batches"] == 1 and auth["dataset"] == cfg["dataset"]
now = datetime.now(ZoneInfo("Asia/Taipei"))
assert now < datetime.fromisoformat(auth["expires_at"]), "STOP: authorization expired"
uses_local = (AGENT_ROOT / "registry/research_authorization_uses.jsonl").read_text()
uses_remote = subprocess.run(["git", "show", f"{VERSION_SYNC['github_remote_head']}:registry/research_authorization_uses.jsonl"],
                             cwd=AGENT_ROOT, text=True, capture_output=True).stdout
assert AUTH_ID not in uses_local and AUTH_ID not in uses_remote, "STOP: GitHub 已有此授權的使用紀錄（1/1）"
prior = prior_archived_attempts(ARCHIVE_ROOT, AUTH_ID)
assert not prior, f"STOP: Drive archive 已有此授權的執行或失敗紀錄 {prior}"
print(f"Authorization {AUTH_ID}: APPROVED by {auth['approved_by']}, expires {auth['expires_at']}, uses 0/1 (local + GitHub + Drive)")
print("Formal gate pin:", auth["formal_gate_pin"]["market_status"], auth["formal_gate_pin"]["gates"])

In [ ]:
# --- 4. Integrity receipt 更新（Core 既有流程）+ 非消耗性 probe ---
from strategy_agent.research_only.development_run import session_config
SESSION_DIR = ARCHIVE_ROOT / "_session_configs" / (datetime.now(ZoneInfo("Asia/Taipei")).strftime("%Y%m%d_%H%M%S_%f") + "_probe")
sc = session_config(cfg_path, SNAPSHOT, SESSION_DIR)
print("session config:", sc["path"], "| canonical", sc["canonical_config_sha256"][:12], "→ session", sc["session_config_sha256"][:12])
print("difference:", sc["difference"])
uses_before = (AGENT_ROOT / "registry/research_authorization_uses.jsonl").read_bytes()
probe = ResearchOnlyRunner(root=AGENT_ROOT).probe(sc["path"], snapshot_root=SNAPSHOT)
assert probe["slot_consumed"] is False and (AGENT_ROOT / "registry/research_authorization_uses.jsonl").read_bytes() == uses_before
print("PROBE:", probe["status"], "| preflight:", probe["preflight"]["status"], "| probe:", probe["probe"]["status"])
for b in probe["blockers"]: print("  BLOCKER", b["code"], b["message"])
print("cost profiles:", probe["profiles"])
assert probe["status"] == "COMPATIBLE_READY_TO_CONSUME" and not probe["blockers"], "STOP: probe 未通過，未消耗額度"
print("\nCOMPATIBLE_READY_TO_CONSUME — 額度仍為 0/1。確認後把第 1 格 I_CONFIRM_SINGLE_BATCH 改成 True。")

In [ ]:
# --- 5. 正式執行：消耗唯一一次研究額度（1/1）---
assert I_CONFIRM_SINGLE_BATCH is True, "未確認：請在第 1 格改成 True，重新執行第 1 格後再執行本格"
assert git("status", "--porcelain", "--untracked-files=all", cwd=AGENT_ROOT) == "", "STOP: checkout 已有變更（可能已執行過）"
from strategy_agent.research_only.development_run import execute_development_smoke, SmokeStopped
RUN_SESSION_DIR = ARCHIVE_ROOT / "_session_configs" / (datetime.now(ZoneInfo("Asia/Taipei")).strftime("%Y%m%d_%H%M%S_%f") + "_run")
context = {"agent_git_sha": AGENT_SHA, "agent_checkout_clean": True, "notebook": NOTEBOOK_VERSION,
           "version_sync": VERSION_SYNC}
RESULT = execute_development_smoke(cfg_path, snapshot_root=SNAPSHOT, archive_root=ARCHIVE_ROOT, session_dir=RUN_SESSION_DIR,
                                   git_commit=AGENT_SHA, actor=ACTOR, context=context, root=AGENT_ROOT)
print("STATUS:", RESULT.get("status"), "| slot consumed:", RESULT.get("slot_consumed"))
if RESULT.get("status") != "COMPLETED":
    print(json.dumps({k: RESULT.get(k) for k in ("status", "error", "code", "blockers", "run_id")}, indent=2, default=str))
    raise SystemExit("執行未完成。依規約停止並回報；不要重跑本格，不要 reset。")
ARCHIVE = Path(RESULT["archive_dir"])
print("ARCHIVE:", ARCHIVE)
print((ARCHIVE / "research_report.md").read_text())

In [ ]:
# --- 6. 把使用紀錄與結果摘要推回 PR #5（不 merge）---
# 只推：registry/research_authorization_uses.jsonl（append 一行）＋ docs/evidence/phase4a_dev_smoke/<run_id>/ 的摘要檔。
import shutil, tempfile, zipfile
assert "RESULT" in dir() and RESULT.get("slot_consumed"), "沒有執行結果可推送"
run_id = Path(RESULT["archive_dir"]).name
use_line = (AGENT_ROOT / "registry/research_authorization_uses.jsonl").read_text().splitlines()[-1]
assert json.loads(use_line)["run_id"] == run_id and json.loads(use_line)["authorization_id"] == AUTH_ID
EVID = ["run_info.txt", "MANIFEST.json", "metrics.json", "cost_breakdown.csv", "execution_diagnostics.json", "research_report.md",
        "trades.csv", "rejected_orders.csv"]
try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    TOKEN = None
if TOKEN:
    pub = Path(tempfile.mkdtemp(prefix="sra_publish_")) / "repo"
    git("clone", "--quiet", "--branch", BRANCH, AGENT_REPO, str(pub))
    uses = pub / "registry/research_authorization_uses.jsonl"
    assert AUTH_ID not in uses.read_text(), "STOP: GitHub 已有使用紀錄；不重複寫入"
    assert uses.read_bytes() == uses_before, "STOP: GitHub 上的使用紀錄檔已變動；請回報，不自動合併"
    with uses.open("a") as f: f.write(use_line + "\n")
    ev = pub / "docs/evidence/phase4a_dev_smoke" / run_id; ev.mkdir(parents=True)
    for n in EVID: shutil.copyfile(ARCHIVE / n, ev / n)
    git("add", "--", "registry/research_authorization_uses.jsonl", str(ev.relative_to(pub)), cwd=pub)
    git("-c", f"user.name={ACTOR}", "-c", f"user.email={ACTOR}@users.noreply.github.com", "commit", "-m",
        f"research-only: consume {AUTH_ID} (1/1) — TSM SMA 5/20 smoke {run_id}", cwd=pub)
    with tempfile.TemporaryDirectory() as d:
        ask = Path(d) / "askpass.sh"
        ask.write_text("#!/bin/sh\ncase \"$1\" in *Username*) printf '%s' x-access-token;; *) printf '%s' \"$GIT_PASSWORD\";; esac\n"); ask.chmod(0o700)
        env = dict(os.environ, GIT_ASKPASS=str(ask), GIT_TERMINAL_PROMPT="0", GIT_PASSWORD=TOKEN)
        r = subprocess.run(["git", "push", "origin", "HEAD:" + BRANCH], cwd=pub, env=env, text=True, capture_output=True)
    TOKEN = None
    assert r.returncode == 0, "push 失敗（exit %d）；請改用下方 zip 上傳，不要重跑第 5 格" % r.returncode
    print("已推送到 PR #5：", git("rev-parse", "HEAD", cwd=pub))
zp = Path("/content") / f"{run_id}_smoke_evidence.zip"
with zipfile.ZipFile(zp, "w") as z:
    z.writestr("registry_use_line.jsonl", use_line + "\n")
    for n in EVID: z.write(ARCHIVE / n, n)
from google.colab import files
files.download(str(zp))
print("Drive archive:", ARCHIVE, "| Authorization uses: 1/1")